<a href="https://colab.research.google.com/github/saeyeon055-hue/Bio-AI-Learning-Path/blob/main/ML/6%EA%B0%95_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRFs를 이용하여 자동 띄어쓰기를 수행하는 프로그램 작성

- 입력(관측): 한글 음절

- 출력(레이블): B, I

   B: 관측된 음절 앞에 공백을 추가해야 함을 나타내는 레이블

   I: 관측된 음절 앞에 공백을 추가하지 말아야 함을 나타내는 레이블

- 데이터 형식
한글 음절 열 \t 레이블 열
예제: 나 는 사 과 가 좋 아 \t B I B I I B I

## google colab 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## CRFs 라이브러리 설치

In [ ]:
!pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.7 MB/s eta 0:00:00


## 데이터 읽기(학습 90%, 평가 10%)

In [ ]:
import os
import sklearn_crfsuite
from sklearn_crfsuite import metrics

In [ ]:
# 파일 경로
file_path = '/content/drive/MyDrive/Colab Notebooks/기계학습/spacing_data.txt'

In [ ]:
# 'spacing_data.txt' 파일을 읽고 lines에 읽은 데이터를 저장
with open(file_path, 'r', encoding='utf8') as inFile:
  lines = inFile.readlines()

In [ ]:
# 데이터를 음절로 이루어진 문장과 정답 값으로 나누어 저장
datas = []
for line in lines:
  pieces = line.strip().split('\t')
  eumjeol_sequence, label = pieces[0].split(), pieces[1].split() # pieces[0]: 음절 열, pieces[1]: 레이블
  datas.append((eumjeol_sequence, label))

number_of_train_datas = int(len(datas) * 0.9)
train_datas = datas[:number_of_train_datas]
test_datas = datas[number_of_train_datas:]

print('train_datas 개수 : ' + str(len(train_datas)))
print('test_datas 개수 : ' + str(len(test_datas)))

train_datas 개수 : 900
test_datas 개수 : 100


## 데이터 변환 (자질 설계)

In [ ]:
def sent2feature(eumjeol_sequence):
  features = []
  sequence_length = len(eumjeol_sequence)
  for index, eumjeol in enumerate(eumjeol_sequence):
    feature = {'BOS':False, 'EOS':False, 'WORD':eumjeol, 'IS_DIGIT':eumjeol.isdigit()}

    if(index == 0):
      feature['BOS'] = True
    elif(index == sequence_length-1):
      feature['EOS'] = True

    if (index-1 >= 0):
      feature['-1_WORD'] = eumjeol_sequence[index-1]
    if (index-2 >= 0):
      feature['-2_WORD'] = eumjeol_sequence[index-2]

    if(index+1 <= sequence_length-1):
      feature['+1_WORD'] = eumjeol_sequence[index+1]
    if(index+2 <= sequence_length-1):
      feature['+2_WORD'] = eumjeol_sequence[index+2]

    features.append(feature)

  return features

In [ ]:
# --- 여기서부터 실제로 실행시키는 코드입니다 ---
input_text = "나는사과가좋아"
result = sent2feature(input_text)

# 결과를 보기 좋게 출력 (첫 번째 글자 '나'에 대한 특징만 예시로 출력)
import pprint
pprint.pprint(result[2]) # 이미지 예시처럼 3번째 글자인 '사'의 결과 출력

{'+1_WORD': '과',
 '+2_WORD': '가',
 '-1_WORD': '는',
 '-2_WORD': '나',
 'BOS': False,
 'EOS': False,
 'IS_DIGIT': False,
 'WORD': '사'}


## 데이터 생성

In [ ]:
train_x, train_y = [], []
for eumjeol_sequence, label in train_datas:
  train_x.append(sent2feature(eumjeol_sequence))
  train_y.append(label)

test_x, test_y = [], []
for eumjeol_sequence, label in test_datas:
  test_x.append(sent2feature(eumjeol_sequence))
  test_y.append(label)

## CRFs 학습

In [ ]:
crf = sklearn_crfsuite.CRF()
crf.fit(train_x, train_y)

CRF()

## CRFs 평가

In [ ]:
def show_predict_result(test_datas, predict):
  for inex_1 in range(len(test_datas)):
    eumjeol_sequence, correct_labels = test_datas[inex_1]
    predict_labels = predict[inex_1]

    correct_sentence, predict_sentence = '', ''
    for index_2 in range(len(eumjeol_sequence)):
      if (index_2== 0):
        correct_sentence += eumjeol_sequence[index_2]
        predict_sentence += eumjeol_sequence[index_2]
        continue

      if (correct_labels[index_2] == 'B'):
        correct_sentence += ' '
      correct_sentence += eumjeol_sequence[index_2]

      if (predict_labels[index_2] == 'B'):
        predict_sentence += ' '
      predict_sentence += eumjeol_sequence[index_2]

    print('정답 문장 : ' + correct_sentence)
    print('예측 문장 : ' + predict_sentence)
    print()

predict = crf.predict(test_x)

In [ ]:
print('Accuracy score : ' + str(metrics.flat_accuracy_score(test_y, predict)))
print()

print('10개의 데이터에 대한 모델 출력과 실제 정답 비교')
print()

show_predict_result(test_datas[:10], predict[:10])

Accuracy score : 0.8964135826020603

10개의 데이터에 대한 모델 출력과 실제 정답 비교

정답 문장 : 1914- 18년의 전쟁은 인류를 통합시킨 최초의 공통분모였다.
예측 문장 : 1914- 18년의 전쟁은 인류를 통합시킨 최초의 공통분 모였다.

정답 문장 : 하지만 이 전쟁은 죽음을 통해 인류를 통합시켰다.
예측 문장 : 하지만이 전쟁은 죽음을 통해 인류를 통합시켰다.

정답 문장 : 사라예보에서 한 세르비아인이 쏜 총 한발이 합스부르크가의 계승자를 죽였다.
예측 문장 : 사라 예보에서 한 세르 비아인이 쏜총한 발이 합스부르크가의 계승 자를 죽였다.

정답 문장 : 이 암살행위는 국지적인 민족주의들과 세계적인 제국주의들이 충돌하는 분쟁지역에서 저질러졌다.
예측 문장 : 이암 살행 위는 국지적인 민족주의 들과 세계적인 제국주의 들이 충돌하는 분쟁 지역에서 저질러졌다.

정답 문장 : 오토만제국의 점진적인 해체는 민족주의의 독기를 발산하는 동시에 오스트리아, 헝가리와 독일, 영국, 프랑스의 탐욕을 자극했다.
예측 문장 : 오토만 제국의 점진 적인 해체는 민족주의의 독기를 발산하는 동시에 오스트리아, 헝가리와 독일, 영국, 프랑스의 탐욕을 자극했다.

정답 문장 : 이렇게 해서 발칸 반도의 한 외진 장소에서 벌어진 국지적인 테러 행위는 일련의 긴박한 반응을 불러 일으키면서 전 유럽에 영향을 미쳤을 뿐만 아니라 이번에는 아시아와 아프리카 식민지들, 일본, 그리고 이어서 미국과 멕시코까지 끌어들였다.
예측 문장 : 이렇게 해서 발칸 반도의 한외진 장소에서 벌어 진국지적인 테러행 위는 일련의 긴박한 반응을 불러일으키면서 전유럽에 영향을 미쳤을 뿐만 아니라이 번에는 아시아와 아프리카 식민지들, 일본, 그리고 이어서 미국과 멕시코까지 끌어들였다.

정답 문장 : 전쟁의 물결이 지구상의 모든 대양으로 밀려드는 동안 캐나다인들과 미국인들, 오스트레일리아인들, 세네갈인들, 알제리인들, 모로코인들, 안남(安南)인들은 연합군 깃발을 휘날리며 유럽전